# YOLO11 Crack Segmentation — COCO → YOLO Pipeline (FIXED)

**All bugs fixed:**
- ✅ RLE segmentation decoded (pycocotools + OpenCV contour → polygon)
- ✅ Category map built from *used* annotations only (skips empty `object` class)
- ✅ Images copied with `shutil.copy2` — no symlinks, works across Drive mountpoints
- ✅ AVIF/WebP/PNG all handled; unsupported formats converted to JPEG
- ✅ All glob patterns updated to include `.webp`, `.avif`, `.png`, etc.
- ✅ `class_names` variable consistent throughout all cells

**Expected COCO folder structure on Drive (images can be flat or in `images/` subfolder):**
```
crack_dataset/
├── train/
│   ├── images/          ← or images directly here
│   └── _annotations.coco.json
├── valid/
│   ├── images/
│   └── _annotations.coco.json
└── test/
    ├── images/
    └── _annotations.coco.json
```

## 1 · Check GPU

In [ ]:
!nvidia-smi

## 2 · Install Dependencies

In [ ]:
# Core dependencies
%pip install -q ultralytics supervision pycocotools

# AVIF support for Pillow (needed if your dataset contains .avif files)
%pip install -q pillow-avif-plugin

# Disable YOLO telemetry
!yolo settings sync=False

import ultralytics
ultralytics.checks()

## 3 · Mount Google Drive + Config

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
HOME = '/content'

# ── USER CONFIG ───────────────────────────────────────────────────────────────
# Path to your unzipped crack_dataset folder (the one containing train/valid/test)
DRIVE_DATASET_PATH = '/content/crack_dataset'   # <-- edit if needed
COCO_JSON_NAME     = '_annotations.coco.json'   # <-- JSON filename inside each split
SPLITS             = ['train', 'valid', 'test']  # <-- split folder names that exist
# ─────────────────────────────────────────────────────────────────────────────

YOLO_DATASET_PATH = os.path.join(HOME, 'crack_yolo')
print('Source dataset :', DRIVE_DATASET_PATH)
print('YOLO output    :', YOLO_DATASET_PATH)

## 4 · Convert COCO Segmentation → YOLO Format

**All three bugs fixed here:**
1. RLE segmentation dicts decoded via `pycocotools.mask.frPyObjects` + OpenCV contours
2. Category map built from *annotated* categories only (drops empty `object` class)
3. Real file copies via `shutil.copy2` — AVIF converted to JPEG automatically

Run the **diagnostic cell** first to confirm paths, then run the **converter cell**.

In [ ]:
# ── DIAGNOSTIC — run this first ──────────────────────────────────────────────
import json
from pathlib import Path

ALL_IMAGE_EXTS = [
    '*.jpg', '*.jpeg', '*.png', '*.webp',
    '*.bmp', '*.tiff', '*.tif', '*.avif',
    '*.JPG', '*.JPEG', '*.PNG', '*.WEBP', '*.AVIF',
]

def glob_images(directory):
    """Glob all image files in a directory regardless of extension."""
    p = Path(directory)
    found = []
    for pat in ALL_IMAGE_EXTS:
        found.extend(p.glob(pat))
    return sorted(set(found))

for split in SPLITS:
    coco_json = Path(DRIVE_DATASET_PATH) / split / COCO_JSON_NAME
    if not coco_json.exists():
        print(f'[MISSING] {coco_json}'); continue

    with open(coco_json) as f:
        coco = json.load(f)

    n_imgs = len(coco['images'])
    n_anns = len(coco['annotations'])
    sample_names = [img['file_name'] for img in coco['images'][:3]]

    # Detect segmentation format
    seg_formats = set()
    for ann in coco['annotations'][:20]:
        seg = ann.get('segmentation')
        if isinstance(seg, dict):
            seg_formats.add('RLE (dict)')
        elif isinstance(seg, list):
            seg_formats.add('Polygon (list)')
        else:
            seg_formats.add(f'Unknown: {type(seg)}')

    # Count annotations per category
    cat_names = {c['id']: c['name'] for c in coco['categories']}
    cat_counts = {}
    for ann in coco['annotations']:
        cid = ann['category_id']
        cat_counts[cid] = cat_counts.get(cid, 0) + 1

    print(f'\n── {split} ({n_imgs} images, {n_anns} anns) ──')
    print(f'  Segmentation format : {seg_formats}')
    print(f'  Sample file_name    : {sample_names}')
    print(f'  Annotations per cat : { {cat_names.get(k, k): v for k, v in cat_counts.items()} }')

    split_root = Path(DRIVE_DATASET_PATH) / split
    for candidate in ['images', '.', 'imgs', 'image']:
        d = split_root / candidate
        if d.is_dir():
            imgs = glob_images(d)
            if imgs:
                print(f'  Images found in     : {d.relative_to(split_root)} ({len(imgs)} files, exts: {sorted({i.suffix.lower() for i in imgs})})')
print()

In [ ]:
# ── CONVERTER — run after diagnostic ─────────────────────────────────────────
import json, os, shutil
import numpy as np
from pathlib import Path
from collections import defaultdict
from PIL import Image as PILImage

# Activate AVIF plugin if available
try:
    import pillow_avif
except ImportError:
    pass

import cv2
from pycocotools import mask as mask_utils


# ── Supported image extensions for YOLO (no conversion needed) ───────────────
YOLO_NATIVE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tiff', '.tif'}
ALL_IMAGE_EXTS   = [
    '*.jpg', '*.jpeg', '*.png', '*.webp',
    '*.bmp', '*.tiff', '*.tif', '*.avif',
    '*.JPG', '*.JPEG', '*.PNG', '*.WEBP', '*.AVIF',
]

def glob_images(directory):
    p = Path(directory)
    found = []
    for pat in ALL_IMAGE_EXTS:
        found.extend(p.glob(pat))
    return sorted(set(found))


# ── Locate source image ───────────────────────────────────────────────────────
def find_image_file(filename: str, search_dirs: list):
    """Find image by exact path, then basename, then recursive search."""
    basename = Path(filename).name
    for d in search_dirs:
        p = Path(d) / filename
        if p.exists(): return p
    for d in search_dirs:
        p = Path(d) / basename
        if p.exists(): return p
    for d in search_dirs:
        hits = list(Path(d).rglob(basename))
        if hits: return hits[0]
    return None


# ── Copy/convert image ────────────────────────────────────────────────────────
def copy_or_convert_image(src: Path, dst_dir: Path, stem: str):
    """
    Copy image to dst_dir using shutil.copy2 (works across Drive mountpoints).
    Unsupported formats (AVIF, etc.) are converted to JPEG via PIL.
    Returns (dst_path, ok).
    """
    ext = src.suffix.lower()
    if ext in YOLO_NATIVE_EXTS:
        dst = dst_dir / (stem + ext)
        if not dst.exists():
            shutil.copy2(src, dst)
        return dst, True
    else:
        # Convert to JPEG
        dst = dst_dir / (stem + '.jpg')
        if not dst.exists():
            try:
                PILImage.open(src).convert('RGB').save(dst, 'JPEG', quality=95)
            except Exception as e:
                print(f'    ⚠️  Could not convert {src.name}: {e}')
                return None, False
        return dst, True


# ── Segmentation → flat polygon ───────────────────────────────────────────────
def seg_to_polygon(seg, img_h: int, img_w: int, min_area: float = 4.0, tolerance: float = 1.0):
    """
    Convert COCO segmentation to a flat polygon [x1, y1, x2, y2, ...].

    Handles:
      - Polygon list  → [[x1,y1,...], ...]  (standard COCO polygon)
      - RLE dict      → {'counts': ..., 'size': [H, W]}  (compressed or uncompressed)

    RLE path: decode mask → find largest external contour → approxPolyDP.
    Returns None if decoding fails or polygon has < 3 points.
    """
    if isinstance(seg, list):
        # Standard polygon format
        if not seg:
            return None
        flat = max(seg, key=len)  # take largest polygon if multiple
        return flat if len(flat) >= 6 else None

    elif isinstance(seg, dict):
        # RLE format (compressed string or uncompressed int list)
        try:
            # frPyObjects handles both compressed string and uncompressed list
            rle          = mask_utils.frPyObjects(seg, img_h, img_w)
            binary_mask  = mask_utils.decode(rle).astype(np.uint8)  # H x W

            contours, _  = cv2.findContours(
                binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
            )
            if not contours:
                return None

            contour = max(contours, key=cv2.contourArea)
            if cv2.contourArea(contour) < min_area:
                return None

            # Approximate to reduce point count while keeping shape
            approx = cv2.approxPolyDP(contour, tolerance, True)
            poly   = approx.flatten().tolist()
            return poly if len(poly) >= 6 else None

        except Exception as e:
            print(f'    ⚠️  RLE decode error: {e}')
            return None

    return None


# ── Build category map from actual annotations ────────────────────────────────
def build_cat_map(dataset_path, splits, json_name):
    """
    Scan all split JSONs to find categories that have at least one annotation.
    Returns (cat_id_to_yolo dict, class_names list).

    Example for crack dataset:
      categories: [{id:0, name:'object'}, {id:1, name:'Crack'}]
      annotations: all use category_id=1
      → cat_id_to_yolo = {1: 0}  → class_names = ['Crack']  → nc=1
    """
    used_ids = set()
    all_cats = {}
    for split in splits:
        p = Path(dataset_path) / split / json_name
        if not p.exists(): continue
        with open(p) as f:
            coco = json.load(f)
        for cat in coco['categories']:
            all_cats[cat['id']] = cat['name']
        for ann in coco['annotations']:
            if not ann.get('iscrowd', 0):
                used_ids.add(ann['category_id'])

    sorted_ids     = sorted(used_ids)
    cat_id_to_yolo = {cid: i for i, cid in enumerate(sorted_ids)}
    class_names    = [all_cats[cid] for cid in sorted_ids]
    return cat_id_to_yolo, class_names


# ── Main converter ────────────────────────────────────────────────────────────
def coco_to_yolo_seg(coco_json_path, split_root, out_labels_dir, out_images_dir, cat_id_to_yolo):
    with open(coco_json_path) as f:
        coco = json.load(f)

    search_dirs = [
        Path(split_root) / 'images',
        Path(split_root),
        Path(split_root) / 'imgs',
        Path(split_root) / 'image',
    ]

    img_meta   = {img['id']: img for img in coco['images']}
    ann_by_img = defaultdict(list)
    for ann in coco['annotations']:
        if not ann.get('iscrowd', 0):
            ann_by_img[ann['image_id']].append(ann)

    Path(out_labels_dir).mkdir(parents=True, exist_ok=True)
    Path(out_images_dir).mkdir(parents=True, exist_ok=True)

    copied = skipped_img = skipped_ann = rle_n = poly_n = 0

    for img_id, img_info in img_meta.items():
        W, H  = img_info['width'], img_info['height']
        fname = img_info['file_name']
        stem  = Path(fname).stem

        src = find_image_file(fname, search_dirs)
        if src is None:
            print(f'    ⚠️  Image not found: {fname}')
            skipped_img += 1
            continue

        dst, ok = copy_or_convert_image(src, Path(out_images_dir), stem)
        if not ok:
            skipped_img += 1
            continue
        copied += 1

        anns = ann_by_img.get(img_id, [])
        label_lines = []

        for ann in anns:
            cls = cat_id_to_yolo.get(ann['category_id'])
            if cls is None:
                skipped_ann += 1
                continue

            seg = ann.get('segmentation')
            if not seg:
                skipped_ann += 1
                continue

            poly = seg_to_polygon(seg, H, W)
            if poly is None:
                skipped_ann += 1
                continue

            # Track format counts for the summary
            if isinstance(seg, dict): rle_n  += 1
            else:                     poly_n += 1

            # Normalise to [0, 1]
            norm = []
            for i in range(0, len(poly) - 1, 2):
                nx = max(0.0, min(1.0, poly[i]     / W))
                ny = max(0.0, min(1.0, poly[i + 1] / H))
                norm.extend([f'{nx:.6f}', f'{ny:.6f}'])

            label_lines.append(f"{cls} {' '.join(norm)}")

        if label_lines:
            (Path(out_labels_dir) / f'{stem}.txt').write_text('\n'.join(label_lines))

    print(f'  ✅ {copied} images copied | '
          f'{skipped_img} not found | '
          f'{skipped_ann} anns skipped | '
          f'{rle_n} RLE + {poly_n} polygon anns OK')


# ── Run ───────────────────────────────────────────────────────────────────────
print('Building category map from annotations...')
cat_id_to_yolo, class_names = build_cat_map(DRIVE_DATASET_PATH, SPLITS, COCO_JSON_NAME)
print(f'  cat_id → yolo_class : {cat_id_to_yolo}')
print(f'  class_names         : {class_names}  (nc={len(class_names)})')
print()

# Wipe stale output
if Path(YOLO_DATASET_PATH).exists():
    shutil.rmtree(YOLO_DATASET_PATH)
    print('Cleared old crack_yolo folder')

for split in SPLITS:
    split_root = Path(DRIVE_DATASET_PATH) / split
    coco_json  = split_root / COCO_JSON_NAME
    if not coco_json.exists():
        print(f'[SKIP] {split}: {coco_json} not found'); continue
    print(f'Converting {split}...')
    coco_to_yolo_seg(
        coco_json_path = coco_json,
        split_root     = split_root,
        out_labels_dir = Path(YOLO_DATASET_PATH) / split / 'labels',
        out_images_dir = Path(YOLO_DATASET_PATH) / split / 'images',
        cat_id_to_yolo = cat_id_to_yolo,
    )

# Final disk count
print()
print('── Final counts on disk ──')
for split in SPLITS:
    img_dir = Path(YOLO_DATASET_PATH) / split / 'images'
    lbl_dir = Path(YOLO_DATASET_PATH) / split / 'labels'
    n_img = len(glob_images(img_dir))           if img_dir.exists() else 0
    n_lbl = len(list(lbl_dir.glob('*.txt')))   if lbl_dir.exists() else 0
    status = '✅' if n_img > 0 and n_lbl > 0 else '⚠️ '
    print(f'  {status} {split}: {n_img} images, {n_lbl} label files')

## 5 · Generate `data.yaml`

In [ ]:
import yaml

# class_names and cat_id_to_yolo were set in Section 4 — don't redefine them here
print('Classes:', class_names, '  nc =', len(class_names))

split_paths = {}
for split in SPLITS:
    p = Path(YOLO_DATASET_PATH) / split / 'images'
    imgs = glob_images(p) if p.exists() else []
    if imgs:
        split_paths[split] = str(p)

yaml_data = dict(
    path  = YOLO_DATASET_PATH,
    train = split_paths.get('train', 'train/images'),
    val   = split_paths.get('valid', 'valid/images'),
    test  = split_paths.get('test',  'test/images'),
    nc    = len(class_names),
    names = class_names,
)

YAML_PATH = os.path.join(YOLO_DATASET_PATH, 'data.yaml')
with open(YAML_PATH, 'w') as f:
    yaml.dump(yaml_data, f, default_flow_style=False, sort_keys=False)

print()
print(open(YAML_PATH).read())

## 6 · Label Sanity Check
Draws converted polygon masks on 3 random training images. If masks look right, proceed to training.

In [ ]:
import random, numpy as np
from pathlib import Path
from PIL import Image as PILImage, ImageDraw
import supervision as sv
from IPython.display import display

# Activate AVIF plugin if available
try:
    import pillow_avif
except ImportError:
    pass

train_img_dir   = Path(YOLO_DATASET_PATH) / 'train' / 'images'
train_label_dir = Path(YOLO_DATASET_PATH) / 'train' / 'labels'

imgs   = glob_images(train_img_dir)
sample = random.sample(imgs, min(3, len(imgs)))

if not sample:
    print('⚠️  No training images found — check Section 4 output')
else:
    for img_path in sample:
        label_path = train_label_dir / (img_path.stem + '.txt')
        if not label_path.exists():
            print(f'No label: {img_path.name}'); continue

        pil = PILImage.open(img_path).convert('RGB')
        W, H = pil.size
        arr  = np.array(pil)

        masks, cls_ids = [], []
        for line in label_path.read_text().strip().splitlines():
            parts = list(map(float, line.split()))
            if len(parts) < 7: continue   # need at least cls + 3 xy pairs
            cls_ids.append(int(parts[0]))
            coords = parts[1:]
            xs = [int(coords[i]     * W) for i in range(0, len(coords), 2)]
            ys = [int(coords[i + 1] * H) for i in range(0, len(coords), 2)]
            m  = PILImage.new('L', (W, H), 0)
            ImageDraw.Draw(m).polygon(list(zip(xs, ys)), fill=1)
            masks.append(np.array(m).astype(bool))

        if not masks:
            print(f'Empty label: {label_path.name}'); continue

        dets = sv.Detections(
            xyxy     = sv.mask_to_xyxy(np.stack(masks)),
            mask     = np.stack(masks),
            class_id = np.array(cls_ids),
        )
        ann = sv.MaskAnnotator(opacity=0.45).annotate(arr.copy(), dets)
        ann = sv.LabelAnnotator(text_color=sv.Color.WHITE).annotate(
            ann, dets, labels=[class_names[c] for c in cls_ids]
        )
        sv.plot_image(ann, size=(8, 8))
        print(f'{img_path.name} — {len(masks)} annotation(s)')

## 7 · Train — `yolo11n-seg.pt` (Fast Sanity Run, ~5 min)

> Skip to **Section 8** if your Section 6 sanity check looks correct.

In [ ]:
%cd {HOME}

!yolo segment train \
    model=yolo11n-seg.pt \
    data={YAML_PATH} \
    epochs=20 \
    imgsz=640 \
    batch=16 \
    patience=10 \
    plots=True \
    project={HOME}/runs/segment \
    name=crack_nano \
    exist_ok=True

In [ ]:
from IPython.display import Image as IPyImage
IPyImage(filename=f'{HOME}/runs/segment/crack_nano/results.png', width=900)

## 8 · Train — `yolo11s-seg.pt` (Full Accuracy Run)

**Crack-optimised hyperparameters:**

| Parameter | Value | Reason |
|---|---|---|
| `imgsz` | 1280 | Cracks are thin — higher resolution captures fine detail |
| `batch` | 8 | Memory-safe at 1280 px on T4 |
| `epochs` | 150 | Crack textures need more exposure to generalise |
| `patience` | 30 | Early-stop if no mAP gain for 30 epochs |
| `close_mosaic` | 15 | Disable mosaic in final 15 epochs for cleaner convergence |
| `hsv_s` | 0.3 | Mild saturation jitter — cracks are mostly greyscale |
| `degrees` | 15 | Cracks appear at any orientation |
| `fliplr/ud` | 0.5 | Both flip directions are valid for cracks |
| `overlap_mask` | True | Required when multiple crack instances overlap |
| `mask_ratio` | 1 | Full-resolution masks — critical for thin crack boundaries |

> **A100/V100:** bump `batch=16` and `imgsz=1536` for even better results.

In [ ]:
%cd {HOME}

!yolo segment train \
    model=yolo11s-seg.pt \
    data={YAML_PATH} \
    epochs=150 \
    imgsz=1280 \
    batch=8 \
    patience=30 \
    mosaic=1.0 \
    close_mosaic=15 \
    hsv_s=0.3 \
    degrees=15.0 \
    fliplr=0.5 \
    flipud=0.5 \
    overlap_mask=True \
    mask_ratio=1 \
    plots=True \
    project={HOME}/runs/segment \
    name=crack_small \
    exist_ok=True

BEST_MODEL = f'{HOME}/runs/segment/crack_small/weights/best.pt'
print('Best weights:', BEST_MODEL)

## 9 · Review Training Results

In [ ]:
from IPython.display import Image as IPyImage, display
import os

for fname in ['results.png', 'confusion_matrix_normalized.png',
              'val_batch0_labels.jpg', 'val_batch0_pred.jpg']:
    path = f'{HOME}/runs/segment/crack_small/{fname}'
    if os.path.exists(path):
        print(f'── {fname} ─────────────')
        display(IPyImage(filename=path, width=900))

## 10 · Validate on Test Set
Reports `mAP50`, `mAP50-95`, Precision, Recall and per-class mask metrics.

In [ ]:
!yolo segment val \
    model={BEST_MODEL} \
    data={YAML_PATH} \
    imgsz=1280 \
    split=test \
    plots=True \
    project={HOME}/runs/segment \
    name=crack_val \
    exist_ok=True

In [ ]:
from IPython.display import Image as IPyImage, display
import os

for fname in ['confusion_matrix_normalized.png', 'val_batch0_pred.jpg']:
    path = f'{HOME}/runs/segment/crack_val/{fname}'
    if os.path.exists(path):
        print(fname)
        display(IPyImage(filename=path, width=900))

## 11 · Predict on Test Images

> Lower `conf` (e.g. 0.15) if cracks are subtle. Raise `iou` (e.g. 0.7) if you see duplicate detections.

In [ ]:
from pathlib import Path

TEST_IMAGES_DIR = str(Path(YOLO_DATASET_PATH) / 'test' / 'images')

!yolo segment predict \
    model={BEST_MODEL} \
    source={TEST_IMAGES_DIR} \
    imgsz=1280 \
    conf=0.25 \
    iou=0.5 \
    save=True \
    save_txt=True \
    project={HOME}/runs/segment \
    name=crack_predict \
    exist_ok=True

## 12 · Visualise Predictions with Supervision

In [ ]:
import random, numpy as np
import supervision as sv
from ultralytics import YOLO
from PIL import Image as PILImage
from IPython.display import display
from pathlib import Path

# Activate AVIF plugin if available
try:
    import pillow_avif
except ImportError:
    pass

model     = YOLO(BEST_MODEL)
test_dir  = Path(YOLO_DATASET_PATH) / 'test' / 'images'
test_imgs = glob_images(test_dir)   # uses glob_images defined in Section 4
sample    = random.sample(test_imgs, min(6, len(test_imgs)))

mask_ann  = sv.MaskAnnotator(opacity=0.45, color_lookup=sv.ColorLookup.CLASS)
box_ann   = sv.BoxAnnotator(thickness=2)
label_ann = sv.LabelAnnotator(text_color=sv.Color.WHITE, text_scale=0.6)

for img_path in sample:
    result = model.predict(str(img_path), imgsz=1280, conf=0.25, iou=0.5, verbose=False)[0]
    dets   = sv.Detections.from_ultralytics(result)
    arr    = np.array(PILImage.open(img_path).convert('RGB'))

    ann = mask_ann.annotate(arr.copy(), dets)
    ann = box_ann.annotate(ann, dets)
    if len(dets):
        labels = [
            f"{class_names[int(c)]} {s:.2f}"
            for c, s in zip(dets.class_id, dets.confidence)
        ]
        ann = label_ann.annotate(ann, dets, labels=labels)

    print(f'{img_path.name}  →  {len(dets)} crack(s) detected')
    sv.plot_image(ann, size=(10, 10))

## 13 · Export to ONNX (Optional)

In [ ]:
from ultralytics import YOLO

model     = YOLO(BEST_MODEL)
onnx_path = model.export(format='onnx', imgsz=1280, simplify=True)
print('ONNX saved to:', onnx_path)

# TensorRT for Jetson / edge — uncomment if needed:
# trt_path = model.export(format='engine', imgsz=1280, half=True)

## 14 · Save Weights to Google Drive

In [ ]:
import shutil, datetime, os

ts        = datetime.datetime.now().strftime('%Y%m%d_%H%M')
drive_out = f'/content/drive/MyDrive/crack_yolo_weights_{ts}'
os.makedirs(drive_out, exist_ok=True)

shutil.copy(BEST_MODEL, f'{drive_out}/best.pt')
shutil.copy(YAML_PATH,  f'{drive_out}/data.yaml')

if 'onnx_path' in dir() and os.path.exists(str(onnx_path)):
    shutil.copy(str(onnx_path), f'{drive_out}/best.onnx')

print('Saved to Drive:', drive_out)
print(os.listdir(drive_out))